# Silver to Gold
## Conversations Notebook - Initial Ingest

### Parameters
Cell 3 reads five notebook widget parameters that control the source and target locations for the entire notebook. Adjust the widget values at the top of the page before running to point at a different catalog, schema, or table.

| Parameter | Default | Description |
|---|---|---|
| `source_catalog` | `silver` | Unity Catalog name for the Silver layer source |
| `source_schema` | `dataverse` | Schema within the source catalog containing the Dataverse tables |
| `target_catalog` | `gold` | Unity Catalog name for the Gold layer destination |
| `target_schema` | `copilot` | Schema in the target catalog (created automatically if absent) |
| `target_table` | `conversations` | Delta table written in the target schema |

Widget defaults can be overridden at runtime via the controls at the top of the page, or by passing job task values when the notebook is scheduled as a job.

In [0]:
source_catalog = dbutils.widgets.get("source_catalog")
source_schema  = dbutils.widgets.get("source_schema")
target_catalog = dbutils.widgets.get("target_catalog")
target_schema  = dbutils.widgets.get("target_schema")
target_table   = dbutils.widgets.get("target_table")

### Load conversation transcript data
Runs a SQL query against the Dataverse Lakehouse to load the `conversationtranscript` table into a Spark DataFrame `ct_df`, selecting key metadata columns and the `content` field, ordered by `conversationstarttime` (most recent first). After the query, the `content` column is wrapped in a single-element `array()` so the downstream JSON parsing cell can uniformly index it as `content[0]`.

In [0]:
from pyspark.sql.functions import col, from_json, schema_of_json, array

query = f"""
SELECT
  id,
  conversationstarttime AS conversation_starttime,
  DATE(conversationstarttime) AS conversation_startdate,
  bot_conversationtranscriptidname,
  bot_conversationtranscriptId,
  content
FROM {source_catalog}.{source_schema}.conversationtranscript
ORDER BY conversationstarttime DESC
"""
ct_df = spark.sql(query)

# Replace content string field with an array field of the same name
ct_df = ct_df.withColumn("content", array(col("content")))

display(ct_df)

### Parse JSON `content` column and flatten top-level fields
Samples the first element (`content[0]`) of the content array from a non-null row to infer the JSON schema. Because Dataverse exports store the JSON as a double-encoded CSV field — the value is wrapped in outer quotes and inner double-quotes are escaped as `""` — the sample is unwrapped before schema inference: leading and trailing `"` characters are stripped and `""` sequences are replaced with `"`.

The inferred schema is applied via `from_json` to `content[0]` across all rows, producing a `content_json` struct column. That struct's top-level fields are then flattened alongside the key transcript metadata columns (`id`, `conversation_starttime`, `conversation_startdate`, `bot_conversationtranscriptidname`, `bot_conversationtranscriptId`) into `parsed_df`.

If no non-null content values are found the code falls back to returning only the transcript metadata columns.

In [0]:
from pyspark.sql.functions import col, from_json, schema_of_json

# 1. Sample one non-null JSON string from the first element of the `content` array to infer schema
sample_row = (
    ct_df
    .where(col("content").isNotNull() & (col("content")[0].isNotNull()))
    .select(col("content")[0].alias("content_element"))
    .limit(1)
    .collect()
)

if not sample_row:
    # No JSON to parse – create an empty DataFrame with same non-JSON columns
    parsed_df = ct_df.select(
        "id",
        "conversation_starttime",
        "conversation_startdate",
        "bot_conversationtranscriptidname",
        "bot_conversationtranscriptId"
    )
else:
    # Remove outer quotes if present and unescape inner quotes
    raw_content = sample_row[0]["content_element"]
    if raw_content.startswith('"') and raw_content.endswith('"'):
        raw_content = raw_content[1:-1].replace('""', '"')

    # 2. Infer schema from the sample JSON
    json_schema = schema_of_json(raw_content)

    # 3. Parse JSON from the first element of the content array
    ct_with_json = ct_df.withColumn(
        "content_json",
        from_json(
            col("content")[0],
            json_schema
        )
    )

    # 4. Flatten JSON fields into top-level columns, keeping original fields as needed
    parsed_df = ct_with_json.select(
        "id",
        "conversation_starttime",
        "conversation_startdate",
        "bot_conversationtranscriptidname",
        "bot_conversationtranscriptId",
        "content_json.*"
    )

# 5. Display the resulting DataFrame with individual JSON fields as columns
display(parsed_df)

### Detect and explode array field for conversation parts
Automatically detects array-type columns in `parsed_df`, chooses the **first** array column as the conversation-parts container, explodes it to create one row per conversation part, filters to only `type == "message"`, and stores the result as `conversation_df` with a `conversation_part_json` struct column.

If no array-type columns are present in `parsed_df`, the code leaves the data unmodified and assigns `conversation_df = parsed_df`.

In [0]:
from pyspark.sql.types import ArrayType
from pyspark.sql.functions import col, explode_outer

# This cell assumes `parsed_df` was created in the previous cell and contains:
# - top-level metadata columns (e.g., id, conversationstarttime, bot_conversationtranscriptidname)
# - flattened JSON columns from `content_json.*` (some of which may still be nested arrays/structs)

# 1. Detect array-type columns that likely hold the per-turn conversation data
array_columns = [
    field.name
    for field in parsed_df.schema.fields
    if isinstance(field.dataType, ArrayType)
]

# If there are no array columns, just show parsed_df as-is
if not array_columns:
    # No nested conversation array detected; nothing to explode
    conversation_df = parsed_df
else:
    # 2. Choose the array column to explode as "conversation parts"
    # If your JSON schema has a specific array column name (e.g. "activities" or "messages"),
    # you can set it explicitly here instead of auto-picking the first one.
    conversation_array_col = array_columns[0]

    # 3. Explode the array so each nested element (conversation turn/part)
    # becomes its own row, while preserving `id` and other metadata
    exploded_df = (
        parsed_df
        .withColumn("conversation_part", explode_outer(col(conversation_array_col)))
    )

    # 4. Keep only those nested elements where conversation_part.type == "message"
    filtered_df = exploded_df.where(col("conversation_part.type") == "message")

    # 5. Keep the nested element as a StructType column instead of converting it to JSON string.
    #    This ensures `conversation_part_json` is NOT a string, but a StructType, and if you later
    #    group/collect it, it can become an ArrayType(StructType()).
    conversation_df = filtered_df.select(
        "id",
        "conversation_starttime",
        "conversation_startdate",
        "bot_conversationtranscriptidname",
        "bot_conversationtranscriptId",
        col("conversation_part").alias("conversation_part_json")
    )

display(conversation_df)


### Extract message-level fields from `conversation_part_json`
Projects key fields out of the `conversation_part_json` struct (channel, text, sender details, timestamp), converts the epoch timestamp to a proper Spark `timestamp`, orders messages by newest first, and produces `conversation_df_with_fields` while keeping the original `conversation_part_json` struct for further exploration if needed.

In [0]:
from pyspark.sql.functions import col, from_unixtime

# conversation_part_json is a STRUCT, so we access its fields directly instead of using get_json_object
conversation_df_with_fields = (
    conversation_df
    .select(
        "id",
        "conversation_starttime",
        "conversation_startdate",
        "bot_conversationtranscriptidname",
        "bot_conversationtranscriptId",
        "conversation_part_json",
        # top-level fields on the struct
        col("conversation_part_json.channelId").alias("channelId"),
        col("conversation_part_json.text").alias("text"),
        col("conversation_part_json.timestamp").cast("double").alias("conversation_part_timestamp"),
        # nested 'from' struct fields
        col("conversation_part_json.from.aadObjectId").alias("from_aadObjectId"),
        col("conversation_part_json.from.id").alias("from_id"),
        col("conversation_part_json.from.role").alias("from_role"),
    )
    # Convert epoch seconds to timestamp (if populated)
    .withColumn(
        "conversation_part_starttime",
        from_unixtime(col("conversation_part_timestamp")).cast("timestamp")
    )
)

# Order the results by the millisecond-based timestamp (most recent first)
conversation_df_with_fields = conversation_df_with_fields.orderBy(col("conversation_part_starttime").desc())

display(conversation_df_with_fields)


### Load system user data
Queries the Dataverse `systemuser` table in the same Lakehouse to retrieve user identity information (AAD object ID, full name, email) into `user_df`. Only rows with a non-null `azureactivedirectoryobjectid` are loaded so that they can be joined reliably with conversation messages later.

In [0]:
userquery = f"SELECT id, azureactivedirectoryobjectid, fullname, internalemailaddress  FROM {source_catalog}.{source_schema}.systemuser where azureactivedirectoryobjectid IS NOT NULL"
user_df = spark.sql(userquery)
display(user_df)

### Join conversations with user details and derive friendly sender name
Renames key user columns in `user_df`, left-joins `conversation_df_with_fields` with user data using the AAD object ID (`from_aadObjectId` ⇔ `userentraid`), and constructs a readable `from` field:
- Bot messages (`from_role == 0`) show the bot name (`bot_conversationtranscriptidname`)
- User messages (`from_role == 1`) show the user full name from Dataverse (`userfullname`).

The code also adds a `from_icon` column (🤖 for bot, 👤 for user) and stores the result in `conversation_with_user`.

In [0]:
from pyspark.sql.functions import col, when, lit

# Rename columns in user_df
user_df_renamed = (
    user_df
    .select(
        col("id").alias("userid"),
        col("azureactivedirectoryobjectid").alias("userentraid"),
        col("fullname").alias("userfullname"),
        col("internalemailaddress").alias("useremailaddress")
    )
)

# Left join conversation_df_simplified to user_df_renamed
conversation_with_user = (
    conversation_df_with_fields.alias("c")
    .join(
        user_df_renamed.alias("u"),
        col("c.from_aadObjectId") == col("u.userentraid"),
        how="left"
    )
    .withColumn(
        "from",
        when(col("c.from_role") == 0, col("c.bot_conversationtranscriptidname"))
        .when(col("c.from_role") == 1, col("u.userfullname"))
    )
    .withColumn(
        "from_icon",
        when(col("c.from_role") == 0, lit("🤖"))
        .when(col("c.from_role") == 1, lit("👤"))
        .otherwise(lit(""))
    )
)

display(conversation_with_user)


In [0]:
# Create the target catalog and schema if they do not already exist.
# toTable() below will create individual Delta tables on first run.
spark.sql(f"CREATE CATALOG IF NOT EXISTS {target_catalog}")
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {target_catalog}.{target_schema}")

print(f"Target schema ready: {target_catalog}.{target_schema}")

### Persist enriched conversation data
**Cell 16** idempotently creates the target catalog and schema in Unity Catalog using `CREATE CATALOG/SCHEMA IF NOT EXISTS` guards so the write cell is safe to run on a clean environment without manual setup.

**Cell 18** writes the `conversation_with_user` DataFrame as a managed Delta table to `{target_catalog}.{target_schema}.{target_table}` (defaults: `gold.copilot.conversations`), overwriting any existing data and permitting schema evolution via `overwriteSchema = true`.

In [0]:
conversation_with_user.write.option("overwriteSchema", "true").mode("overwrite").saveAsTable(f"`{target_catalog}`.`{target_schema}`.`{target_table}`")